In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

# nb_img_05: MLP Pseudo-Labeling — Config A

Self-training MLP on frozen CLIP embeddings.  
Config A: `threshold=0.9`, `hidden_sizes=(256, 128)`

In [2]:
import numpy as np
import pandas as pd
import json
from sklearn.metrics import f1_score
from utils.image_config import IMAGE_MANIFEST_PATH, IMAGE_EMBEDDINGS_PATH, IMAGE_CLASS_NAMES, IMAGE_SEED, RESULTS_DIR
from utils.image_pseudo_labeling import pseudo_label_loop

manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
embeddings = np.load(IMAGE_EMBEDDINGS_PATH)

seed_mask = manifest["is_seed"].to_numpy().astype(bool)
seed_labels = manifest.loc[seed_mask, "label_idx"].to_numpy()
true_labels = manifest["label_idx"].to_numpy()

print(f"Seed: {seed_mask.sum()}  Unlabeled: {(~seed_mask).sum()}")

Seed: 275  Unlabeled: 5225


In [3]:
pred_labels, history = pseudo_label_loop(
    embeddings=embeddings,
    seed_mask=seed_mask,
    seed_labels=seed_labels,
    hidden_sizes=(256, 128),
    threshold=0.9,
    max_iters=10,
    seed=IMAGE_SEED,
    n_classes=len(IMAGE_CLASS_NAMES),
)
print("History:", history)
final_coverage = (pred_labels != -1).mean()
label_acc = (pred_labels == true_labels).mean()
label_f1 = f1_score(true_labels, pred_labels, average="macro", zero_division=0)
print(f"Config A — ACC: {label_acc:.4f}  F1: {label_f1:.4f}  Coverage: {final_coverage:.1%}")

  Iter 0: +4291 pseudo-labels  coverage=83.0%


  Iter 1: +806 pseudo-labels  coverage=97.7%


  Iter 2: +77 pseudo-labels  coverage=99.1%


  Iter 3: +9 pseudo-labels  coverage=99.2%


  Iter 4: +8 pseudo-labels  coverage=99.4%


...

  Iter 6: +1 pseudo-labels  coverage=99.5%


  Iter 7: +1 pseudo-labels  coverage=99.5%


  Iter 8: +5 pseudo-labels  coverage=99.6%


  Iter 9: +3 pseudo-labels  coverage=99.7%


History: [{'iteration': 0, 'new_pseudo_labels': 4291, 'cumulative_coverage': 0.8301818181818181}, {'iteration': 1, 'new_pseudo_labels': 806, 'cumulative_coverage': 0.9767272727272728}, {'iteration': 2, 'new_pseudo_labels': 77, 'cumulative_coverage': 0.9907272727272727}, {'iteration': 3, 'new_pseudo_labels': 9, 'cumulative_coverage': 0.9923636363636363}, {'iteration': 4, 'new_pseudo_labels': 8, 'cumulative_coverage': 0.9938181818181818}, {'iteration': 5, 'new_pseudo_labels': 6, 'cumulative_coverage': 0.9949090909090909}, {'iteration': 6, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9950909090909091}, {'iteration': 7, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9952727272727273}, {'iteration': 8, 'new_pseudo_labels': 5, 'cumulative_coverage': 0.9961818181818182}, {'iteration': 9, 'new_pseudo_labels': 3, 'cumulative_coverage': 0.9967272727272727}]
Config A — ACC: 0.9935  F1: 0.9935  Coverage: 100.0%


In [4]:
out_df = manifest[["filepath", "true_label"]].copy()
out_df["predicted_label"] = [IMAGE_CLASS_NAMES[p] for p in pred_labels]
RESULTS_DIR.mkdir(exist_ok=True)
out_df.to_csv(RESULTS_DIR / "image_full_labels_pseudo_labeling_A.csv", index=False)

metrics = {
    "config": "A",
    "hidden_sizes": [256, 128],
    "threshold": 0.9,
    "label_acc": float(label_acc),
    "label_f1": float(label_f1),
    "coverage": float(final_coverage),
    "history": history,
}
with open(RESULTS_DIR / "image_metrics_pseudo_labeling_A.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

{'config': 'A', 'hidden_sizes': [256, 128], 'threshold': 0.9, 'label_acc': 0.9934545454545455, 'label_f1': 0.993455138429218, 'coverage': 1.0, 'history': [{'iteration': 0, 'new_pseudo_labels': 4291, 'cumulative_coverage': 0.8301818181818181}, {'iteration': 1, 'new_pseudo_labels': 806, 'cumulative_coverage': 0.9767272727272728}, {'iteration': 2, 'new_pseudo_labels': 77, 'cumulative_coverage': 0.9907272727272727}, {'iteration': 3, 'new_pseudo_labels': 9, 'cumulative_coverage': 0.9923636363636363}, {'iteration': 4, 'new_pseudo_labels': 8, 'cumulative_coverage': 0.9938181818181818}, {'iteration': 5, 'new_pseudo_labels': 6, 'cumulative_coverage': 0.9949090909090909}, {'iteration': 6, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9950909090909091}, {'iteration': 7, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9952727272727273}, {'iteration': 8, 'new_pseudo_labels': 5, 'cumulative_coverage': 0.9961818181818182}, {'iteration': 9, 'new_pseudo_labels': 3, 'cumulative_coverage': 0.99672727